In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ── Load Data ──────────────────────────────────────────────
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

if 'Survived' in test.columns:
    test = test.drop(columns=['Survived'])

test_ids = test['PassengerId'].copy()
dfs = [train, test]

# ── Feature Engineering ────────────────────────────────────
for df in dfs:
    df['Title'] = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
    rare = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
    df['Title']      = df['Title'].replace(rare, 'Rare')
    df['Title']      = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
    df['SmallFamily']= ((df['FamilySize'] >= 2) & (df['FamilySize'] <= 4)).astype(int)
    df['LargeFamily']= (df['FamilySize'] >= 5).astype(int)
    df['HasCabin']   = df['Cabin'].notna().astype(int)
    df['WomanOrChild']= ((df['Sex'] == 'female') | (df['Age'] < 12)).astype(int)

# ── Fill Missing Values ────────────────────────────────────
age_medians  = train.groupby('Title')['Age'].median()
fallback_age = age_medians.get('Rare', age_medians.median())

for df in dfs:
    df['Age'] = df.apply(
        lambda row: age_medians.get(row['Title'], fallback_age)
        if pd.isnull(row['Age']) else row['Age'], axis=1)
    df['Fare']     = df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
    df['Embarked'] = df['Embarked'].fillna(train['Embarked'].mode()[0])
    df['AgeBand']  = pd.cut(df['Age'],  bins=[0,12,18,35,60,100], labels=[0,1,2,3,4]).astype(int)
    df['FareBand'] = pd.qcut(df['Fare'], q=4, labels=[0,1,2,3]).astype(int)
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']
    df['Pclass_Sex']    = df['Pclass'] * (df['Sex'] == 'male').astype(int)

# ── Encode ─────────────────────────────────────────────────
le_title = LabelEncoder()
le_title.fit(pd.concat([train['Title'], test['Title']]))
train['Title'] = le_title.transform(train['Title'])
test['Title']  = le_title.transform(test['Title'])

for col in ['Sex', 'Embarked']:
    le = LabelEncoder()
    le.fit(train[col])
    train[col] = le.transform(train[col])
    test[col]  = le.transform(test[col])

# ── Features ───────────────────────────────────────────────
FEATURES = ['Pclass', 'Sex', 'AgeBand', 'FareBand', 'FarePerPerson',
            'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin',
            'WomanOrChild', 'Pclass_Sex', 'LargeFamily', 'SmallFamily']

X      = train[FEATURES]
y      = train['Survived']
X_test = test[FEATURES]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Define Models (pre-tuned, no GridSearch) ───────────────
models = {
    'XGBoost'   : XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                 subsample=0.8, colsample_bytree=0.8,
                                 eval_metric='logloss', random_state=42, verbosity=0),

    'LightGBM'  : LGBMClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=42, verbose=-1),

    'RF'        : RandomForestClassifier(n_estimators=300, max_depth=6,
                                          min_samples_split=4, random_state=42),

    'ExtraTrees': ExtraTreesClassifier(n_estimators=300, max_depth=6,
                                        min_samples_split=4, random_state=42),

    'GBM'       : GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                              learning_rate=0.05, random_state=42),

    'LR'        : LogisticRegression(max_iter=500, C=0.1, random_state=42),
}

# ── Cross Validate Each Model ──────────────────────────────
print(f'{"Model":<15}  {"CV Accuracy":>12}  {"Std":>8}')
print('-' * 40)

cv_scores = {}
for name, model in models.items():
    s = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    cv_scores[name] = s.mean()
    print(f'{name:<15}  {s.mean():.4f}        ±{s.std():.4f}')

# ── Train All Models on Full Data ──────────────────────────
print('\nTraining on full dataset...')
proba_list = []

for name, model in models.items():
    model.fit(X, y)
    proba = model.predict_proba(X_test)[:, 1]
    proba_list.append((name, cv_scores[name], proba))
    print(f'  {name} trained ✅')

# ── Weighted Average by CV Score ───────────────────────────
# Models with higher CV scores get more weight automatically
total_weight = sum(score for _, score, _ in proba_list)
weighted_proba = sum(
    (score / total_weight) * proba
    for _, score, proba in proba_list
)

final_predictions = (weighted_proba >= 0.5).astype(int)

# ── Submission ─────────────────────────────────────────────
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived'   : final_predictions
})
submission.to_csv('submission.csv', index=False)

print(f'\nDone!')
print(f'Predicted {final_predictions.sum()} survivors out of {len(final_predictions)}')

# ── Quick Weight Summary ───────────────────────────────────
print('\nModel Weights in Final Ensemble:')
for name, score, _ in proba_list:
    weight = score / total_weight
    print(f'  {name:<15}  weight={weight:.3f}  (cv={score:.4f})')

Model             CV Accuracy       Std
----------------------------------------
XGBoost          0.8372        ±0.0168
LightGBM         0.8383        ±0.0220
RF               0.8328        ±0.0077
ExtraTrees       0.8294        ±0.0106
GBM              0.8339        ±0.0205
LR               0.8171        ±0.0187

Training on full dataset...
  XGBoost trained ✅
  LightGBM trained ✅
  RF trained ✅
  ExtraTrees trained ✅
  GBM trained ✅
  LR trained ✅

Done!
Predicted 154 survivors out of 418

Model Weights in Final Ensemble:
  XGBoost          weight=0.168  (cv=0.8372)
  LightGBM         weight=0.168  (cv=0.8383)
  RF               weight=0.167  (cv=0.8328)
  ExtraTrees       weight=0.166  (cv=0.8294)
  GBM              weight=0.167  (cv=0.8339)
  LR               weight=0.164  (cv=0.8171)
